# Fingerprint localization from first principles
**Objective:** build a fingerprint matrix, calculate neighbors, select k using validation, fit and diagnose a public-data localizer.
**Data:** Antwerp public derivative (12,000 packets, 39 gateway columns); source details in `data/README.md`.
**Prerequisites:** notebook 00 and array indexing.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


## 1. Calculate a miniature prediction
These three examples are illustrative, not experimental observations. Feature distances use dBm coordinates; output distances use meters.

In [ ]:
reference_X = np.array([[-80.,-100.],[-84.,-96.],[-110.,-75.]])
reference_Y = np.array([[0.,0.],[100.,0.],[0.,100.]])
query = np.array([-82.,-98.])
distance = np.linalg.norm(reference_X-query,axis=1)
nearest = np.argsort(distance)[:2]
weights = 1/distance[nearest]
position = np.average(reference_Y[nearest],axis=0,weights=weights)
display(pd.DataFrame({'distance':distance,'selected':np.isin(np.arange(3),nearest)}))
print('Prediction:',position,'Error:',np.linalg.norm(position-[60.,10.]))
np.testing.assert_allclose(position,[50.,0.])


## 2. Inspect the actual columns before fitting
GPS is the response. Identifiers and timestamps audit the partition. A missing RSSI receives a fixed code here, not an asserted receiver sensitivity.

In [ ]:
d = pd.read_csv(ROOT/'data/raw/antwerp/antwerp_localization.csv')
features = [c for c in d if c.startswith('rssi_')]
X = d[features].fillna(-140.).to_numpy()
R = 6371000.; origin = (51.22,4.40)
y = np.column_stack([R*np.cos(np.deg2rad(origin[0]))*np.deg2rad(d.lon-origin[1]), R*np.deg2rad(d.lat-origin[0])])
tr = np.flatnonzero(d.split.eq('treino'))
va = np.flatnonzero(d.split.isin(['validacao','validação']))
te = np.flatnonzero(d.split.eq('teste'))
assert len(tr)==8000 and len(va)==2000 and len(te)==2000
display(d[['timestamp','lat','lon']+features[:3]].head())
print('Observed gateway count:',d[features].notna().sum(axis=1).describe())


## 3. Select k without test outcomes
We report a geographic distance. The conversion is explicitly evaluated in float64. Validation is used to choose k; the test is used after that choice.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from experiments import haversine

def geographic_error(true_xy,pred_xy):
    true_xy,pred_xy = np.asarray(true_xy,dtype=float),np.asarray(pred_xy,dtype=float)
    def latlon(xy):
        return (origin[0]+np.rad2deg(xy[:,1]/R), origin[1]+np.rad2deg(xy[:,0]/(R*np.cos(np.deg2rad(origin[0])))))
    return haversine(*latlon(true_xy),*latlon(pred_xy))

candidates=[]
for k in [3,9,21]:
    fitted=KNeighborsRegressor(n_neighbors=k,weights='distance',n_jobs=2).fit(X[tr],y[tr])
    score=geographic_error(y[va],fitted.predict(X[va])).mean()
    candidates.append((score,k,fitted))
display(pd.DataFrame([(s,k) for s,k,_ in candidates],columns=['validation_mean_error_m','k']))
score,k,model=min(candidates,key=lambda z:z[0])
error=geographic_error(y[te],model.predict(X[te]))
display(pd.Series({'selected_k':k,'mean_m':error.mean(),'median_m':np.median(error),'p90_m':np.quantile(error,.9)}))


## 4. Add a tree comparator and inspect generalization
The forest receives the same test rows and RSSI columns. Its fixed settings make this an illustrative comparison, not an equal-budget family benchmark.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor(n_estimators=100,min_samples_leaf=5,random_state=42,n_jobs=2).fit(X[tr],y[tr])
rf_error=geographic_error(y[te],rf.predict(X[te]))
print('RF mean error m:',rf_error.mean())
archived=pd.read_csv(MATCHED/'metrics.csv')
display(archived[archived.case.str.startswith('antwerp_')].groupby(['case','model']).mean_error_m.agg(['mean','std']))


## Exercise and worked interpretation
**Exercise:** why does a mean-plus-mask representation change nearest neighbors even though the raw measurements are unchanged? Why does a temporal test not necessarily evaluate new places?

<details><summary>Feedback after your attempt</summary>

imputation and mask coordinates alter the distance geometry. The mask's scale relative to dBm matters. A later packet may come from a previously observed location; a spatial partition asks another question. The diagnostic changes are conditional on their specific representation and partition.

</details>

**Adaptation task:** reserve a set of geographic cells before tuning, rebuild the gateway vocabulary, and explain how a previously unseen gateway will be handled. Do not select the split by its resulting error.